<a href="https://colab.research.google.com/github/samarkun23/RLMs_from_scratch/blob/main/RLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import requests
import json
import re
import sys
import traceback
from contextlib import redirect_stdout, redirect_stderr
from google.colab import userdata

# API Key from Google Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found in Google Colab Secrets!"
    )

# Models
ROOT_MODEL = "gemini-2.5-flash"
SUB_MODEL = "gemini-2.5-flash"


def llm_call(prompt, system="", model=ROOT_MODEL, max_tokens=4000):
    url = (
        "https://generativelanguage.googleapis.com/v1beta/"
        f"models/{model}:generateContent"
    )

    headers = {
        "x-goog-api-key": GEMINI_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "contents": [
            {
                "role": "user",
                "parts": [{"text": prompt}]
            }
        ],
        "generationConfig": {
            "maxOutputTokens": max_tokens,
            "thinkingConfig": {
                "thinkingBudget": 0
            }
        }
    }

    if system:
        payload["systemInstruction"] = {
            "parts": [{"text": system}]
        }

    try:
        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=(10, 60)
        )

        response.raise_for_status()
        data = response.json()

        candidates = data.get("candidates", [])

        if not candidates:
            raise RuntimeError(
                f"No candidates returned: {json.dumps(data)[:1500]}"
            )

        candidate = candidates[0]
        parts = candidate.get("content", {}).get("parts", [])

        result = "".join(
            part.get("text", "")
            for part in parts
        )

        if not result:
            reason = candidate.get("finishReason", "Unknown")
            raise RuntimeError(
                f"No text returned. Finish reason: {reason}"
            )

        return result

    except requests.exceptions.Timeout as e:
        raise RuntimeError(
            f"Gemini API timed out for model {model}."
        ) from e

    except requests.exceptions.HTTPError as e:
        error_body = response.text[:1500]
        raise RuntimeError(
            f"Gemini API HTTP error {response.status_code}: "
            f"{error_body}"
        ) from e


# Test the function
answer = llm_call(
            "Say hello in one word",
            max_tokens=100
        )
print("Gemini response:", answer)



Gemini response: Hi


In [2]:
!pip -q install -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


# Part 1:The Problem -- Long context Fails
Let's create a task that easy for humans but hard for LLM at scale: **Count specific items burried in a long, noisy dataset**;

In [3]:
import random
random.seed(42)

first_names = [
    "Alice", "Bob", "Charlie", "David", "Emma",
    "Frank", "Grace", "Henry", "Isabella", "Jack",
    "Liam", "Olivia", "Noah", "Sophia", "Mason",
    "Ava", "Ethan", "Mia", "Lucas", "Amelia",
    "James", "Charlotte", "Benjamin", "Evelyn", "Daniel"
]

cities = [
    "London", "Tokyo", "New York", "Paris", "Berlin",
    "Mumbai", "Delhi", "Dubai", "Singapore", "Sydney",
    "Toronto", "Barcelona", "Rome", "Seoul", "Amsterdam"
]

professions = [
    "Software Engineer",
    "Doctor",
    "Teacher",
    "Designer",
    "Data Scientist",
    "Lawyer",
    "Accountant",
    "Architect",
    "Photographer",
    "Marketing Manager"
]

def generate_dataset(n_entries=500):
  """ Generate data set of people with names , cities and professions."""
  entries=[]
  for i in range(n_entries):
    name=f"{random.choice(first_names)}{random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')}"
    city = random.choice(cities)
    prof = random.choice(professions)
    age = random.randint(22,65)
    entries.append(f"Entry {i+1}: {name}, age {age}, {prof}, based in {city}")
  return entries

entries = generate_dataset(5000)
dataset_text = "\n".join(entries)

ground_truth = sum(1 for e in entries if "Engineer" in e and "Tokyo" in e)

print(f"Dataset: {len(entries)} entries {len(dataset_text)} characters")
print(f"\nFirst 5 entries")
for e in entries[:5]:
  print(f" {e}")
print(f"\n Ground truth : {ground_truth} enginners in Tokyo")

Dataset: 5000 entries 284043 characters

First 5 entries
 Entry 1: JamesD, age 37, Data Scientist, based in London
 Entry 2: HenryE, age 65, Doctor, based in Barcelona
 Entry 3: EvelynR, age 49, Marketing Manager, based in Tokyo
 Entry 4: BobA, age 36, Designer, based in Tokyo
 Entry 5: EthanT, age 34, Photographer, based in London

 Ground truth : 19 enginners in Tokyo


# Try it with a vanilla LLM -- just stuff the context in

In [4]:
vanilla_prompt = f"""
Count exactly how many Software Engineers are based in Tokyo.

Dataset:
{dataset_text}

Return ONLY the final integer. Do not explain or list entries.
"""

vanilla_answer = llm_call(vanilla_prompt, max_tokens=100)

print("Vanilla LLM answer:", vanilla_answer)
print("Ground truth:", ground_truth)

# Extract the answer only if it is a single integer
answer = vanilla_answer.strip()

is_correct = answer.isdigit() and int(answer) == ground_truth

print(f"Correct: {'Yes' if is_correct else 'No'}")

Vanilla LLM answer: 40
Ground truth: 19
Correct: No


# Part 2 Build with REPL Environment

In [5]:
import io

In [7]:
class RLMRepl:
  def __init__(self,context:str, max_output_chars: int = 5000):
    self.final_answer = None
    self.max_output_chars = max_output_chars
    self.sub_count_call = 0

    self.namespace = {
        "context": context, # <-- the long input lives here
        "FINAL": self.final, # call FINAL("answer") to finish
        "llm_query": self.llm_query, #call sub-llm (recursion)
        # standard lib modules the model might want .
        "re": re,
        "json": json,
        "len": len,
        "print": print,
        "int": int,
        "float": float,
        "str": str,
        "list": list,
        "dict": dict,
        "range": range,
        "enumerate": enumerate,
        "sum": sum,
        "sorted": sorted,
        "min": min,
        "max": max,
        "abs": abs,
        "set": set,
        "zip": zip,
        "tuple":tuple,
        "map": map,
        "filter": filter,
        "isinstance": isinstance,
        "type": type,
        "True": True,
        "False": False,
        "None": None,
    }

  def final(self,answer):
    self.final_answer = str(answer)
    print(f"[FINAL ANSWER SUBMITTED: {answer}]")

  ## subllm
  def llm_query(self,query:str,sub_context:str ="") -> str:
    self.sub_count_call +=1
    print(f"[Sub-LLM call #{self.sub_count_call}: '{query[:80]}...")

    prompt = query
    if sub_context:
      prompt = f"{query}\n\nContext:\n{sub_context}"

    result = llm_call(prompt, model=SUB_MODEL, max_tokens=2000)
    print(f"Sub-LLM Returned: '{result[:100]}...']")
    return result

  def execute(self, code:str) -> str:
    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()

    try:
      with redirect_stdout(stdout_capture), redirect_stderr(stderr_capture):
        exec(code, self.namespace)
      output = stdout_capture.getvalue()
    except Exception as e:
      output = f"ERROR: {type(e).__name__}: {e}"

    if len(output) > self.max_output_chars:
      output = output[:self.max_output_chars] + f"\n... [truncated to {self.max_output_chars} chars]"

    return output

repl = RLMRepl(context="Hello world! This is a test context with some data.")
print("Test 1 - peek at context:")
print(repl.execute('print(context[:30])'))

print("\nTest2 - search with regex: ")
print(repl.execute('import re\nmatches = re.findall(r")'))

print("\nTest 3 - submit final ans")
print(repl.execute('FINAL(42)'))
print(f"final answer stored: {repl.final_answer}")

print("REPL WORKS")


Test 1 - peek at context:
Hello world! This is a test co


Test2 - search with regex: 
ERROR: SyntaxError: unterminated string literal (detected at line 2) (<string>, line 2)

Test 3 - submit final ans
[FINAL ANSWER SUBMITTED: 42]

final answer stored: 42
REPL WORKS


# Build the RLM Agent Loop
The core loop:
1. Give the LLM the query + system prompt(but Not the context)
2. LLM write the python code
3. Execute the code in the REPL ( where context lives )
4. Send the output back to the LLM
5. Repeat util FINAL() is called or we hit max iterations

In [9]:
RLM_SYSTEM_PROMPT="""You are an RLM (Recursive Language Model) agent.

You have access to a Python REPL environment. The user's data is stored in a variable called 'context' - it may be very long (millions of characters). You CANNOT see the context directly. You must write Python code to explore it.

Available tools:
- `context` - the full input text (Python string variable)
- `print()` - use this to see output from your code
- `11m_query(query, sub_context)` - call a sub-LLM to analyze a chunk.
 The sub-LLM's response is returned as a string. It does NOT enter your context
- `FINAL (answer)` - call this when you have the final answer.
- Standard Python: 're', 'json', 'len', 'sum, etc.

Strategy:
1. First, check the size: `print(len(context))`
2. Peek at the structure:`print(context[:500])`
3. Use code to search, filter, count, or slice the data
4. For complex subtasks, use `llm_query()` to delegate to a sub-LLM
5. When done, call 'FINAL (your_answer)`


Rules:
- Write ONLY Python code. No markdown, no explanation.
- Your code block must be wrapped in ```python...```
- Use print() to see results you only see what you print.
- Variables persist between steps (like Jupyter cells).
- Be systematic. Explore first, then solve.
"""

print("System prompt defined")

System prompt defined


In [20]:
# THE RLM LOOP
def extract_code(response:str) -> str:
  """Extract Python code from the LLM's response."""
  # Try to find ``python ... blocks
  pattern = r'```python\s*\n(.*?)```'
  matches = re.findall (pattern, response, re.DOTALL)
  if matches:
    return matches[0].strip()
  # Try```...```blocks
  pattern = r'```\s*\n(.*?)```'
  matches = re.findall (pattern, response, re.DOTALL)
  if matches:
      return matches[0].strip()

  # If no code blocks, try to use the whole response as code
  # (strip any leading text before first line that looks like code)
  lines = response.strip().split('\n')
  code_lines = []
  started = False
  for line in lines:
    if not started and line.startswith(('import ', 'from ', 'print(', '#', 'for', 'if', 'def', 'context', 'result', 'count')):
       started = True
    if started:
      code_lines.append(line)

  return '\n'.join(code_lines) if code_lines else response.strip()


def run_llm(
    query: str,
    context: str,
    max_iterations: int = 10,
    verbose: bool = True
):
    repl = RLMRepl(context)
    history = []

    user_msg = f"""
Task: {query}

The data is stored in the `context` variable.
It contains {len(context)} characters.

You cannot see the context directly. Use Python code
to explore it through the REPL.

Start by checking the structure and size of the data.
Use print() to inspect results.
When you have the final answer, call FINAL(answer).
"""

    history.append(f"USER TASK:\n{user_msg}")

    for i in range(max_iterations):

        if verbose:
            print(f"\n{'=' * 60}")
            print(f"ITERATION {i + 1}/{max_iterations}")
            print(f"{'=' * 60}")

        # Build prompt from the conversation so far
        prompt = "\n\n".join(history)

        prompt += """

Now write the NEXT Python code to execute in the REPL.

Remember:
- The context variable already exists.
- Variables persist between iterations.
- Use print() to inspect data.
- Use llm_query() only when useful.
- Call FINAL(answer) when the task is complete.
- Return ONLY executable Python code inside a
  ```python code block.
"""

        # Call your existing Gemini API function
        try:
            assistant_msg = llm_call(
                prompt=prompt,
                system=RLM_SYSTEM_PROMPT,
                model=ROOT_MODEL,
                max_tokens=4000
            )

        except Exception as e:
            if verbose:
                print(f"Gemini API error: {e}")
            break

        # Extract Python code from Gemini's response
        code = extract_code(assistant_msg)

        if verbose:
            print("\nLLM wrote:\n")
            print(code)

        # Execute code inside the REPL
        output = repl.execute(code)

        if verbose:
            print("\nREPL output:\n")
            print(output if output else "(No output)")

        # Save the model response and execution result
        history.append(
            f"ASSISTANT CODE:\n{assistant_msg}"
        )

        history.append(
            f"REPL EXECUTION OUTPUT:\n{output}"
        )

        # Check whether FINAL() was called
        if repl.final_answer is not None:
            if verbose:
                print("\n" + "=" * 60)
                print("FINAL ANSWER")
                print("=" * 60)
                print(repl.final_answer)
                print(
                    f"\nSub-LLM calls: {repl.sub_count_call}"
                )

            return repl.final_answer

    if verbose:
        print("\nMaximum iterations reached or execution stopped.")

    return repl.final_answer


In [29]:
answer = run_llm(
    query="Count how many Software Engineers are based in Tokyo.",
    context=dataset_text,
    max_iterations=5,
    verbose=True
)

print("Final answer:", answer)


ITERATION 1/5

LLM wrote:

print(len(context))
print(context[:500])

REPL output:

284043
Entry 1: JamesD, age 37, Data Scientist, based in London
Entry 2: HenryE, age 65, Doctor, based in Barcelona
Entry 3: EvelynR, age 49, Marketing Manager, based in Tokyo
Entry 4: BobA, age 36, Designer, based in Tokyo
Entry 5: EthanT, age 34, Photographer, based in London
Entry 6: BenjaminU, age 48, Photographer, based in Barcelona
Entry 7: HenryO, age 22, Data Scientist, based in Sydney
Entry 8: DanielZ, age 43, Accountant, based in New York
Entry 9: IsabellaE, age 28, Lawyer, based in Paris
Ent


ITERATION 2/5

LLM wrote:

lines = context.strip().split('\n')
tokyo_software_engineers = 0
for line in lines:
    if "Software Engineer" in line and "based in Tokyo" in line:
        tokyo_software_engineers += 1
print(tokyo_software_engineers)

REPL output:

19


ITERATION 3/5

LLM wrote:

FINAL(tokyo_software_engineers)

REPL output:

[FINAL ANSWER SUBMITTED: 19]


FINAL ANSWER
19

Sub-LLM calls: 0
F